In [ ]:
import pandas as pd
import json

## 1. Load the flattened PubMedQA data

In [ ]:
df = pd.read_csv("combined_output.csv")

# Rows without a question, context, or long_answer can't be converted meaningfully
df = df.dropna(subset=["id", "question", "contexts", "long_answer"])

print(f"Loaded {len(df)} usable rows.")
df.head()

In [ ]:
def convert_to_squad(dataframe):
    squad_format = {
        "version": "v2.0",
        "data": []
    }

    matched = 0
    unmatched = 0

    for _, row in dataframe.iterrows():
        qid = str(row["id"])
        question = str(row["question"]).strip()
        context = str(row["contexts"]).replace(" || ", " ").strip()
        answer_text = str(row["long_answer"]).strip()

        answer_start = context.find(answer_text)

        if answer_start == -1 or not answer_text:
            answers = []
            is_impossible = True
            unmatched += 1
        else:
            answers = [{
                "text": answer_text,
                "answer_start": answer_start
            }]
            is_impossible = False
            matched += 1

        squad_format["data"].append({
            "title": "PubMedQA",
            "paragraphs": [{
                "context": context,
                "qas": [{
                    "id": qid,
                    "question": question,
                    "answers": answers,
                    "is_impossible": is_impossible
                }]
            }]
        })

    total = matched + unmatched
    print(f"Converted {total} examples.")
    print(f"  Matched (answerable):   {matched} ({matched/total:.1%})")
    print(f"  Unmatched (impossible): {unmatched} ({unmatched/total:.1%})")

    return squad_format

In [ ]:
squad_data = convert_to_squad(df)

with open("data.json", "w") as f:
    json.dump(squad_data, f, indent=2)

print("Saved SQuAD-format file as data.json")

In [ ]:
with open("data.json") as f:
    data_json = json.load(f)

all_ids = []

for entry in data_json["data"]:
    for para in entry["paragraphs"]:
        for qa in para["qas"]:
            assert "id" in qa and isinstance(qa["id"], str), f"Missing or non-string ID: {qa}"
            assert "question" in qa, f"Missing question for ID: {qa.get('id')}"
            assert "answers" in qa and isinstance(qa["answers"], list), f"Invalid answers for ID: {qa.get('id')}"
            for ans in qa["answers"]:
                assert "text" in ans and "answer_start" in ans, f"Invalid answer format: {ans}"
            all_ids.append(qa["id"])

print(f"Schema valid. Found {len(all_ids)} questions.")
print("Sample IDs:", all_ids[:5])